# Evaluate DocNLC model

Notebook nay dung de evaluate checkpoint tren mot tap anh co GT. Luong chay tuong duong `test.py`: doc filelist `degraded|gt`, load model, chay `model.test()`, luu anh restore, tinh PSNR/SSIM.

Luu y: `model.test()` cua `SIEN_model.py` dang threshold output o muc `0.95`, giong logic test goc cua repo.

## 1. Setup project

In [ ]:
from pathlib import Path
import os
import sys
import math
import logging
import random
import yaml

import cv2
import numpy as np
import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "test.py").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "test.py").exists():
            PROJECT_ROOT = parent
            break

assert (PROJECT_ROOT / "test.py").exists(), "Khong tim thay root repo DocNLC. Hay mo notebook tu thu muc DocNLC."
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    for idx in range(torch.cuda.device_count()):
        print(f"GPU {idx}:", torch.cuda.get_device_name(idx))

## 2. Tao file `path.txt`

Filelist evaluate can co format:

```text
/path/to/degraded.png|/path/to/gt.png
```

Cell duoi day co the tao filelist tu 2 folder co ten file trung stem, hoac ghi tu list cap anh thu cong.

In [ ]:
IMAGE_EXTS = {".bmp", ".jpg", ".jpeg", ".png", ".tif", ".tiff"}

def image_map(folder):
    folder = Path(folder).expanduser().resolve()
    assert folder.is_dir(), f"Khong tim thay folder: {folder}"
    files = sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
    mapping = {}
    for path in files:
        if path.stem in mapping:
            raise ValueError(f"Trung stem {path.stem} trong {folder}")
        mapping[path.stem] = path
    return mapping

def write_pair_filelist_from_folders(degraded_dir, gt_dir, output_txt, strict=True):
    degraded = image_map(degraded_dir)
    gt = image_map(gt_dir)
    lines = []
    missing = []
    for stem, degraded_path in degraded.items():
        gt_path = gt.get(stem)
        if gt_path is None:
            missing.append(stem)
            continue
        lines.append(f"{degraded_path}|{gt_path}")
    if strict and missing:
        raise RuntimeError(f"Thieu GT cho {len(missing)} anh. Vi du: {missing[:10]}")
    output_txt = Path(output_txt).expanduser().resolve()
    output_txt.parent.mkdir(parents=True, exist_ok=True)
    output_txt.write_text("\n".join(lines) + ("\n" if lines else ""))
    print(f"Da ghi {len(lines)} dong vao {output_txt}")
    if missing:
        print(f"Bo qua {len(missing)} anh bi thieu GT. Vi du:", missing[:10])
    return output_txt

def write_pair_filelist_from_pairs(pairs, output_txt):
    lines = []
    for degraded_path, gt_path in pairs:
        degraded_path = Path(degraded_path).expanduser().resolve()
        gt_path = Path(gt_path).expanduser().resolve()
        assert degraded_path.exists(), f"Khong ton tai degraded: {degraded_path}"
        assert gt_path.exists(), f"Khong ton tai GT: {gt_path}"
        lines.append(f"{degraded_path}|{gt_path}")
    output_txt = Path(output_txt).expanduser().resolve()
    output_txt.parent.mkdir(parents=True, exist_ok=True)
    output_txt.write_text("\n".join(lines) + ("\n" if lines else ""))
    print(f"Da ghi {len(lines)} dong vao {output_txt}")
    return output_txt

# Cach 1: tao path.txt tu 2 folder degraded/gt co ten file trung nhau.
WRITE_FROM_FOLDERS = False
DEGRADED_DIR = PROJECT_ROOT / "eval_data" / "degraded"
GT_DIR = PROJECT_ROOT / "eval_data" / "gt"
EVAL_FILELIST = PROJECT_ROOT / "path.txt"

if WRITE_FROM_FOLDERS:
    EVAL_FILELIST = write_pair_filelist_from_folders(DEGRADED_DIR, GT_DIR, EVAL_FILELIST, strict=True)

# Cach 2: tao path.txt tu list cap anh thu cong.
# MANUAL_PAIRS = [
#     (PROJECT_ROOT / "TestImages" / "degraded.png", PROJECT_ROOT / "TestImages" / "gt.png"),
# ]
# EVAL_FILELIST = write_pair_filelist_from_pairs(MANUAL_PAIRS, PROJECT_ROOT / "path.txt")

print("EVAL_FILELIST:", EVAL_FILELIST)

## 3. Nhap cau hinh evaluate

In [ ]:
# Sua cac bien nay truoc khi chay evaluate.
EVAL_FILELIST = PROJECT_ROOT / "path.txt"
PRETRAIN_MODEL_G = PROJECT_ROOT / "pretrain" / "model.pth"

EXPERIMENT_NAME = "DocNLC_eval_notebook"
OUTPUT_ROOT = PROJECT_ROOT / "output"
GPU_IDS = [0]

# Giu batch size = 1 vi model.test() tu xu ly patch theo tung anh.
BATCH_SIZE = 1
PATCH_SIZE = 256
STRICT_LOAD = False

## 4. Validate filelist va checkpoint

In [ ]:
def preview_filelist(filelist, expected_cols=2, max_lines=5):
    filelist = Path(filelist).expanduser().resolve()
    assert filelist.exists(), f"Khong tim thay filelist: {filelist}"
    lines = [line.strip() for line in filelist.read_text().splitlines() if line.strip()]
    assert lines, f"Filelist dang rong: {filelist}"
    for idx, line in enumerate(lines[:100], 1):
        cols = line.split("|")
        if len(cols) != expected_cols:
            raise ValueError(f"Dong {idx} co {len(cols)} cot, can {expected_cols}: {line}")
        for path in cols:
            if not Path(path).expanduser().exists():
                raise FileNotFoundError(f"Dong {idx} co path khong ton tai: {path}")
    print(f"Filelist: {filelist} ({len(lines)} dong)")
    for line in lines[:max_lines]:
        print("  ", line)
    return filelist, len(lines)

EVAL_FILELIST, NUM_IMAGES = preview_filelist(EVAL_FILELIST)
PRETRAIN_MODEL_G = Path(PRETRAIN_MODEL_G).expanduser().resolve()
assert PRETRAIN_MODEL_G.exists(), f"Khong tim thay checkpoint: {PRETRAIN_MODEL_G}"
print("Checkpoint:", PRETRAIN_MODEL_G)

## 5. Sinh YAML tam

In [ ]:
config = {
    "name": EXPERIMENT_NAME,
    "use_tb_logger": False,
    "model": "sr",
    "distortion": "sr",
    "scale": 1,
    "gpu_ids": GPU_IDS,
    "datasets": {
        "val": {
            "name": "UEN",
            "mode": "UEN_test",
            "filelist": str(EVAL_FILELIST),
            "batch_size": BATCH_SIZE,
            "use_shuffle": False,
            "IN_size": PATCH_SIZE,
        }
    },
    "network_G": {
        "which_model_G": "SID",
        "nf": 16,
        "groups": 8,
    },
    "path": {
        "root": str(OUTPUT_ROOT),
        "results_root": str(OUTPUT_ROOT),
        "pretrain": str(PROJECT_ROOT / "pretrain1"),
        "pretrain_model_G": str(PRETRAIN_MODEL_G),
        "strict_load": STRICT_LOAD,
        "resume_state": None,
    },
    "train": {
        "lr_G": 1e-4,
        "lr_scheme": "MultiStepLR",
        "beta1": 0.9,
        "beta2": 0.99,
        "niter": 1,
        "ewc": False,
        "distill": False,
        "ewc_coff": 50.0,
        "distill_coff": 0.2,
        "fix_some_part": None,
        "warmup_iter": -1,
        "ComputeImportance": False,
        "istraining": False,
        "lr_steps": [60, 90],
        "lr_gamma": 0.5,
        "eta_min": 5e-6,
        "pixel_criterion": "l1",
        "pixel_weight": 5000.0,
        "ssim_weight": 1000.0,
        "vgg_weight": 1000.0,
        "val_epoch": 1,
        "manual_seed": 0,
    },
    "logger": {
        "print_freq": 40,
        "save_checkpoint_epoch": 25,
    },
}

generated_dir = PROJECT_ROOT / "options" / "generated"
generated_dir.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = generated_dir / f"{EXPERIMENT_NAME}.yml"
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True))
print("Da ghi config:", CONFIG_PATH)
print(CONFIG_PATH.read_text())

## 6. Load dataset va model

In [ ]:
import options.options as option
from utils import util
from data import create_dataloader, create_dataset
from models import create_model

opt = option.parse(str(CONFIG_PATH), is_train=False)
opt["dist"] = False
opt["train"]["distill"] = False
opt["train"]["ewc"] = False
if not torch.cuda.is_available():
    opt["gpu_ids"] = None

results_root = Path(opt["path"]["results_root"])
test_img_dir = results_root / "test_images"
test_img_dir.mkdir(parents=True, exist_ok=True)

util.set_random_seed(opt["train"]["manual_seed"] or 0)
torch.backends.cudnn.benchmark = True

val_set = create_dataset(opt, opt["datasets"]["val"])
val_loader = create_dataloader(val_set, opt["datasets"]["val"], opt, None)
opt = option.dict_to_nonedict(opt)

model = create_model(opt)
print("Val images:", len(val_set))
print("Output dir:", test_img_dir)

## 7. Evaluate

In [ ]:
psnr_values = []
ssim_values = []
rows = []

def load_gt_for_metrics(gt_path, lq_path, output_shape):
    gt_img = cv2.imread(str(gt_path), cv2.IMREAD_COLOR)
    lq_img = cv2.imread(str(lq_path), cv2.IMREAD_COLOR)
    if gt_img is None:
        raise FileNotFoundError(f"Khong doc duoc GT: {gt_path}")
    if lq_img is None:
        raise FileNotFoundError(f"Khong doc duoc degraded image: {lq_path}")
    if gt_img.shape != lq_img.shape:
        raise ValueError(f"Cap LQ/GT nguon khong cung shape: LQ={lq_img.shape}, GT={gt_img.shape}")
    if gt_img.shape != output_shape:
        gt_img = cv2.resize(gt_img, (output_shape[1], output_shape[0]))
    return gt_img

for idx, val_data in enumerate(val_loader, 1):
    img_name = val_data["LQ_path"][0]
    gt_name = val_data["GT_path"][0]
    model.feed_data(val_data)
    model.test()

    visuals = model.get_current_visuals()
    en_img = np.clip(visuals["rlt"], 0, 255).astype(np.uint8)
    gt_img = load_gt_for_metrics(gt_name, img_name, en_img.shape)

    out_name = Path(img_name).name
    cv2.imwrite(str(test_img_dir / out_name), en_img)

    psnr = util.calculate_psnr(en_img, gt_img)
    ssim = util.calculate_ssim(en_img, gt_img)
    rows.append({"image": out_name, "psnr": psnr, "ssim": ssim})

    if not (math.isinf(psnr) or math.isnan(psnr)):
        psnr_values.append(psnr)
        ssim_values.append(ssim)

    print(f"[{idx}/{len(val_loader)}] {out_name} PSNR={psnr:.4f} SSIM={ssim:.4f}")

avg_psnr = sum(psnr_values) / len(psnr_values) if psnr_values else float("nan")
avg_ssim = sum(ssim_values) / len(ssim_values) if ssim_values else float("nan")
print("\nAverage PSNR:", round(avg_psnr, 4))
print("Average SSIM:", round(avg_ssim, 4))
print("Saved restored images to:", test_img_dir)

## 8. Luu metrics CSV

In [ ]:
import csv

metrics_csv = results_root / "metrics.csv"
with metrics_csv.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["image", "psnr", "ssim"])
    writer.writeheader()
    writer.writerows(rows)

print("Metrics CSV:", metrics_csv)